# 01 - Basil Segmentation Dataset Analysis

This notebook is a thin wrapper around `src/data` and `src/visualization`.
It scaffolds the dataset directories, generates the metadata template,
validates images/masks, builds the dataset report, and previews a few
samples with their mask overlays.

Run this in Google Colab or locally. No GPU is required for this notebook.

In [ ]:
# If running in Google Colab, clone/mount the repository first, then:
# %pip install -r requirements.txt

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src.utils.config import load_config

config = load_config(PROJECT_ROOT / "configs" / "segmentation.yaml")
print(config)

## Step 1: Scaffold directories and metadata template

Place raw images under `data/basil/segmentation/images/` and matching
binary masks under `data/basil/segmentation/masks/` before running this.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts" / "prepare_dataset.py"),
     "--config", str(PROJECT_ROOT / "configs" / "segmentation.yaml")],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)

## Step 2: Validate the dataset and build the report

In [ ]:
from src.data.report import build_dataset_report, save_dataset_report
from src.data.validation import DatasetValidator
from src.utils.io_utils import find_image_files

validator = DatasetValidator(
    images_dir=config.dataset.images_dir,
    masks_dir=config.dataset.masks_dir,
)
validation_result = validator.validate()

num_image_files = len(find_image_files(config.dataset.images_dir))
num_mask_files = len(find_image_files(config.dataset.masks_dir))

report = build_dataset_report(validation_result, num_image_files, num_mask_files)
output_paths = save_dataset_report(
    report, validation_result,
    output_dir=PROJECT_ROOT / config.experiment.output_dir / "dataset_report",
)
print(report)
print("Errors:", len(validation_result.errors))
print("Warnings:", len(validation_result.warnings))

## Step 3: Preview sample images, masks, and overlays

In [ ]:
from src.utils.io_utils import safe_read_image, safe_read_mask
from src.visualization.dataset_preview import plot_leaf_coverage_histogram, plot_sample_grid
from pathlib import Path as _Path

images_dir = _Path(config.dataset.images_dir)
masks_dir = _Path(config.dataset.masks_dir)

samples = []
for stats in validation_result.per_image_stats[:6]:
    image = safe_read_image(images_dir / stats.image_filename)
    mask = safe_read_mask(masks_dir / stats.mask_filename)
    samples.append({"image": image, "mask": mask, "filename": stats.image_filename})

if samples:
    plot_sample_grid(samples)
    leaf_coverages = [item.leaf_coverage for item in validation_result.per_image_stats]
    plot_leaf_coverage_histogram(leaf_coverages)
else:
    print("No valid image/mask pairs available yet to preview.")